# Week 1 — Introduction to Embeddings

**Phase 01: Embeddings & Semantic Similarity for Pension Fund Analytics**

---

## What are Embeddings?

Think of a world map. Every city has a latitude and longitude — two numbers that encode its position such that *geographically close cities have numerically close coordinates*. Amsterdam and Rotterdam are 60 km apart, so their coordinates are similar. Amsterdam and Tokyo are 9,000 km apart, so their coordinates differ wildly.

**Embeddings do the same thing for meaning.**

A text embedding is a list of numbers (a *vector*) that positions a piece of text in high-dimensional space such that texts with **similar meaning** end up **close together** — regardless of which exact words they use.

| Concept | Map analogy | Embedding analogy |
|---|---|---|
| Representation | (lat, lon) | 384-dimensional vector |
| Closeness metric | km distance | cosine similarity |
| Similar items | nearby cities | documents about the same topic |
| Learned from | geography | billions of text pairs |

The key insight: **the model has learned that 'coverage ratio' and 'dekkingsgraad' (Dutch) refer to the same concept**, so their vectors land near each other even though they share zero characters.

## Why Embeddings for Pension Fund Documents?

Pension fund analysts and regulators deal with a specific problem: the same concept appears under many names.

**Example — solvency:**
- Dutch regulation (FTK): *dekkingsgraad*, *beleidsdekkingsgraad*, *minimaal vereist eigen vermogen*
- IORP II (EU): *technical provisions*, *own funds*, *solvency capital requirement*
- Academic literature: *coverage ratio*, *funding ratio*, *asset-liability gap*
- Practitioner shorthand: *CR*, *funding level*, *underfunding*

A keyword search for `"coverage ratio"` returns **zero results** from a Dutch DNB report that uses *dekkingsgraad* throughout. A regulator searching for IORP compliance articles misses every FTK equivalent.

**Semantic search solves this.** Once we embed the entire corpus, a query for *"pension fund minimum capital requirements"* retrieves documents about FTK Article 131, IORP II Directive Title IV, *and* actuarial papers on stochastic ALM — because they all cluster together in embedding space.

This is why we're building a semantic search engine over pension fund research: to let analysts ask questions in plain language and find relevant documents across language barriers, jargon variation, and regulatory frameworks.

In [ ]:
# ── Cell 1: Install and verify dependencies ──────────────────────────────────
# If running in a fresh environment, uncomment and run:
# !pip install sentence-transformers numpy pandas scikit-learn matplotlib seaborn

import importlib
import sys

required = {
    'sentence_transformers': 'sentence-transformers',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
}

missing = []
for module, pkg in required.items():
    try:
        importlib.import_module(module)
        print(f'  OK  {pkg}')
    except ImportError:
        print(f'  MISSING  {pkg}  <- run: pip install {pkg}')
        missing.append(pkg)

if missing:
    print(f'\nInstall missing packages then restart the kernel.')
else:
    print('\nAll dependencies satisfied.')

In [ ]:
# ── Cell 2: Load the sentence-transformers model ──────────────────────────────
# all-MiniLM-L6-v2:
#   - 22M parameters, 384-dimensional output vectors
#   - Trained on 1B+ sentence pairs via contrastive learning
#   - ~80 MB download, cached to ~/.cache/huggingface after first load
#   - Fast enough to embed 500 abstracts in < 10 seconds on CPU

from sentence_transformers import SentenceTransformer
import numpy as np

MODEL_NAME = 'all-MiniLM-L6-v2'
print(f'Loading model: {MODEL_NAME}')
model = SentenceTransformer(MODEL_NAME)

# Inspect model architecture summary
print(f'\nMax sequence length : {model.max_seq_length} tokens')
print(f'Embedding dimension : {model.get_sentence_embedding_dimension()}')
print(f'\nModel loaded and ready.')

In [ ]:
# ── Cell 3: Your first embedding ─────────────────────────────────────────────
# Encode a single sentence and inspect the raw vector.

sentence = "Dutch pension funds must maintain a coverage ratio above 110% under FTK regulations."

embedding = model.encode(sentence)  # returns numpy array, shape (384,)

print(f'Input  : "{sentence}"')
print(f'\nVector shape    : {embedding.shape}')
print(f'Vector dtype    : {embedding.dtype}')
print(f'First 8 values  : {np.round(embedding[:8], 4)}')
print(f'\nL2 norm (magnitude) : {np.linalg.norm(embedding):.4f}')
print()
print('Note: the raw vector is NOT unit-normalised by default.')
print('We will normalise explicitly when computing cosine similarity.')
print(f'Max value  : {embedding.max():.4f}')
print(f'Min value  : {embedding.min():.4f}')
print(f'Mean value : {embedding.mean():.4f}')

In [ ]:
# ── Cell 4: Semantic similarity — the magic moment ───────────────────────────
# Compute cosine similarity between sentence pairs.
# The key test: Dutch 'dekkingsgraad' should score HIGH vs 'coverage ratio'.

from sklearn.metrics.pairwise import cosine_similarity

pairs = [
    # (sentence_a, sentence_b, expected: high or low similarity)
    (
        "The coverage ratio of the pension fund fell below the minimum threshold.",
        "De dekkingsgraad van het pensioenfonds daalde onder het minimumvereiste.",
        "HIGH — same concept, different language"
    ),
    (
        "ESG integration in pension fund investment strategies.",
        "Environmental and social criteria in long-term asset allocation.",
        "HIGH — same topic, different phrasing"
    ),
    (
        "The IORP II directive governs occupational pension schemes across the EU.",
        "Stochastic modelling of interest rate term structures.",
        "LOW — pension regulation vs. quantitative finance"
    ),
    (
        "Liability-driven investing reduces duration mismatch in pension portfolios.",
        "LDI strategies match asset duration to pension liability duration.",
        "HIGH — LDI, nearly paraphrases"
    ),
]

print(f'{"Pair":<4} {"Similarity":<12} {"Expected"}')
print('-' * 65)

for i, (a, b, expected) in enumerate(pairs, 1):
    emb_a = model.encode([a])
    emb_b = model.encode([b])
    sim = cosine_similarity(emb_a, emb_b)[0][0]
    print(f'#{i:<3} {sim:.4f}       {expected}')
    print(f'     A: "{a[:70]}"')
    print(f'     B: "{b[:70]}"')
    print()

In [ ]:
# ── Cell 5: Batch embedding — load articles.csv and embed all abstracts ───────
import pandas as pd
import os
import time

DATA_PATH = os.path.join('..', 'data', 'raw', 'articles.csv')

# ── Load data (with graceful fallback if file not yet present) ───────────────
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f'Loaded {len(df)} articles from {DATA_PATH}')
    print(f'Columns: {list(df.columns)}')
    print(f'Categories: {df["category"].value_counts().to_dict()}')
    print(f'Years: {df["year"].min()} – {df["year"].max()}')
else:
    # Create a small synthetic dataset so the notebook still runs
    print(f'WARNING: {DATA_PATH} not found. Using synthetic data for demonstration.')
    print('Run the data generation script first, or place articles.csv in data/raw/')
    print()
    SYNTHETIC_DATA = [
        {'id': 1, 'title': 'FTK Coverage Ratio Requirements', 'abstract': 'The Dutch FTK framework requires pension funds to maintain a policy coverage ratio above 110%. Funds below the minimum required coverage ratio of 90% must submit a recovery plan to DNB.', 'category': 'pension_regulation', 'year': 2022, 'authors': 'De Vries, J.', 'source_journal': 'Pensioen Magazine', 'word_count': 180},
        {'id': 2, 'title': 'ESG Integration in ALM', 'abstract': 'We examine how ESG criteria can be incorporated into asset-liability management frameworks for European pension funds without materially degrading funding ratio stability.', 'category': 'investment_theory', 'year': 2023, 'authors': 'Schmidt, A.', 'source_journal': 'Journal of Pension Economics', 'word_count': 210},
        {'id': 3, 'title': 'IORP II Implementation Across Member States', 'abstract': 'The IORP II directive introduces new governance, risk management and transparency requirements for occupational pension institutions across EU member states.', 'category': 'pension_regulation', 'year': 2021, 'authors': 'Müller, H.', 'source_journal': 'European Insurance Law Review', 'word_count': 195},
        {'id': 4, 'title': 'Transformer Models for Financial Text Classification', 'abstract': 'We fine-tune BERT on a corpus of financial regulatory documents to classify sections by topic. The model outperforms TF-IDF baselines on pension regulation text.', 'category': 'fintech_ai', 'year': 2023, 'authors': 'Liu, Y.', 'source_journal': 'FinNLP Workshop', 'word_count': 220},
        {'id': 5, 'title': 'Interest Rate Risk in Defined Benefit Plans', 'abstract': 'Duration mismatch between pension liabilities and fixed-income assets creates significant interest rate exposure. Liability-driven investing strategies reduce this gap.', 'category': 'actuarial', 'year': 2022, 'authors': 'Johnson, R.', 'source_journal': 'North American Actuarial Journal', 'word_count': 175},
        {'id': 6, 'title': 'Machine Learning for Credit Risk', 'abstract': 'Gradient boosting models trained on loan-level data outperform logistic regression for PD estimation. Feature engineering from macroeconomic indicators improves stability.', 'category': 'general_ml', 'year': 2023, 'authors': 'Park, S.', 'source_journal': 'Journal of Risk', 'word_count': 200},
        {'id': 7, 'title': 'Inflation and Pension Liability Valuation', 'abstract': 'Rising inflation affects both the real value of pension payments and the discount rate used to value liabilities under different regulatory regimes.', 'category': 'macroeconomics', 'year': 2023, 'authors': 'Bakker, T.', 'source_journal': 'Macro Finance Review', 'word_count': 185},
        {'id': 8, 'title': 'Stochastic ALM for Pension Funds', 'abstract': 'We develop a stochastic asset-liability model for Dutch pension funds that captures equity, interest rate, and inflation risks under the FTK framework.', 'category': 'actuarial', 'year': 2022, 'authors': 'Van den Berg, M.', 'source_journal': 'Insurance: Mathematics and Economics', 'word_count': 250},
    ]
    df = pd.DataFrame(SYNTHETIC_DATA)
    print(f'Created {len(df)} synthetic articles for demonstration.')

print(f'\nSample row:')
print(df.iloc[0][['title', 'category', 'year', 'abstract']].to_string())

In [ ]:
# ── Cell 6: Embed all abstracts and time the operation ───────────────────────
# normalize_embeddings=True divides each vector by its L2 norm,
# so dot product == cosine similarity (faster search later)

abstracts = df['abstract'].tolist()

print(f'Embedding {len(abstracts)} abstracts with {MODEL_NAME}...')
t0 = time.time()

embeddings = model.encode(
    abstracts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,   # unit-normalise each vector
)

elapsed = time.time() - t0
print(f'\nDone in {elapsed:.1f}s  ({len(abstracts)/elapsed:.0f} abstracts/sec)')
print(f'\nEmbedding matrix shape : {embeddings.shape}')
print(f'  Rows = {embeddings.shape[0]} articles')
print(f'  Cols = {embeddings.shape[1]} embedding dimensions')
print(f'Memory usage : {embeddings.nbytes / 1024:.1f} KB')
print(f'\nVerify unit norm (should all be 1.0):')
norms = np.linalg.norm(embeddings, axis=1)
print(f'  min={norms.min():.6f}  max={norms.max():.6f}  mean={norms.mean():.6f}')

In [ ]:
# ── Cell 7: 2D visualisation with PCA ────────────────────────────────────────
# Reduce 384 dimensions to 2 for plotting.
# PCA is deterministic and fast; it preserves the most variance.
# (For larger corpora, UMAP or t-SNE can reveal finer cluster structure.)

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(embeddings)

explained = pca.explained_variance_ratio_
print(f'PCA explained variance:')
print(f'  PC1: {explained[0]*100:.1f}%')
print(f'  PC2: {explained[1]*100:.1f}%')
print(f'  Total: {sum(explained)*100:.1f}%')
print()
print('Note: PCA captures a small fraction of the total variance in 384-D space.')
print('The plot shows rough clusters; UMAP would show tighter structure.')

# ── Build plot dataframe ──────────────────────────────────────────────────────
plot_df = pd.DataFrame({
    'PC1': coords_2d[:, 0],
    'PC2': coords_2d[:, 1],
    'category': df['category'],
    'title': df['title'],
})

# ── Colour palette ────────────────────────────────────────────────────────────
categories = plot_df['category'].unique()
palette = sns.color_palette('tab10', n_colors=len(categories))
colour_map = dict(zip(categories, palette))

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))

for cat in categories:
    mask = plot_df['category'] == cat
    ax.scatter(
        plot_df.loc[mask, 'PC1'],
        plot_df.loc[mask, 'PC2'],
        label=cat,
        color=colour_map[cat],
        s=80,
        alpha=0.8,
        edgecolors='white',
        linewidths=0.5,
    )

ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}% variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({explained[1]*100:.1f}% variance)', fontsize=11)
ax.set_title(
    'Pension Research Corpus — Abstract Embeddings (PCA 2D)\n'
    f'Model: {MODEL_NAME}  |  {len(df)} articles',
    fontsize=12
)
ax.legend(title='Category', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('week1_pca_plot.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved to week1_pca_plot.png')

In [ ]:
# ── Cell 8: Save embeddings for use in Week 2 ────────────────────────────────
import os

EMBED_PATH = 'article_embeddings.npy'
META_PATH  = 'article_index_meta.csv'

np.save(EMBED_PATH, embeddings)
df.to_csv(META_PATH, index=False)

print(f'Saved embeddings  -> {EMBED_PATH}  ({os.path.getsize(EMBED_PATH)/1024:.0f} KB)')
print(f'Saved metadata    -> {META_PATH}')
print()
print('These files are used by:')
print('  - week2_similarity_search.ipynb  (nearest-neighbour search)')
print('  - week4_semantic_search_engine.ipynb  (ChromaDB ingestion)')
print('  - semantic_search_cli.py  (the capstone build project)')

## Key Takeaways

1. **Embeddings are coordinates for meaning** — the 384-dimensional vector for "dekkingsgraad" sits close to the vector for "coverage ratio" because the model has seen them used in the same contexts across training data.

2. **The model is a function: text → vector** — `model.encode(text)` is the only API you need. Batch it for efficiency.

3. **Normalisation matters** — setting `normalize_embeddings=True` gives unit vectors, making cosine similarity equivalent to a simple dot product (important for Week 2).

4. **PCA visualisation shows rough clusters** — `pension_regulation` and `actuarial` documents cluster together; `general_ml` sits further away. This is geometry, not labels.

5. **384 dimensions × 500 articles = ~750 KB** — entirely in-memory, no database needed at this scale.

---

## Exercises

1. **Encode your own pairs.** Find two DNB supervisory reports and one unrelated document. Do the similarity scores match your intuition?

2. **Change the model.** Replace `all-MiniLM-L6-v2` with `BAAI/bge-small-en-v1.5`. Does the PCA cluster structure change? Do the cross-lingual similarity scores improve?

3. **Visualise with UMAP.** Install `umap-learn` and replace PCA with UMAP. Compare the cluster separation. (Hint: `umap.UMAP(n_components=2, random_state=42).fit_transform(embeddings)`)

4. **Longest abstract first.** Sort `df` by `word_count` descending. What happens to the embedding time for very long abstracts? What does the model do when an abstract exceeds 256 tokens?

5. **Category centroids.** Compute the mean embedding for each category. Which two categories have the highest centroid cosine similarity? Does that match your domain intuition?